# Task 1 — Preparazione del dataset (Bank Marketing)

> 📄 Documentazione completa e motivazioni: [`docs/task1.md`](../docs/task1.md)

## 0. Import e riproducibilità

Importiamo le librerie di base (`pandas` e `numpy`) e fissiamo un **seme casuale** (`SEED = 10`).
Tutto il progetto usa lo stesso seme: serve a rendere i risultati **riproducibili**, cioè a
garantire che il campionamento casuale delle istanze per `manuale.csv` produca sempre lo stesso
sottoinsieme a ogni esecuzione. La riproducibilità è un requisito metodologico fondamentale
quando si presentano risultati sperimentali.

In [1]:
import pandas as pd
import numpy as np

In [2]:
SEED = 10
np.random.seed(SEED)

## 1. Caricamento del dataset

Carichiamo il dataset originale **Bank Marketing** dalla cartella `data/raw/`. Il file usa il
**punto e virgola** (`;`) come separatore di campo, secondo il formato distribuito dal UCI
Repository: lo specifichiamo con `sep=";"`. Il dataset contiene circa **41.000 istanze** e
**21 colonne** (20 attributi + la variabile target `y`).

In [3]:
PATH = "../data/raw/bank-additional-full.csv"
df = pd.read_csv(PATH, sep=";")

In [4]:
print(f"Caricato dataset: {df.shape[0]} istanze x {df.shape[1]} attributi")

Caricato dataset: 41188 istanze x 21 attributi


In [5]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Controllo qualità di base
### 2a. Duplicati

In [6]:
n_dup = df.duplicated().sum()
print(f"Istanze duplicate trovate: {n_dup}")

Istanze duplicate trovate: 12


In [7]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset dopo la rimozione: {df.shape[0]} istanze")

Dataset dopo la rimozione: 41176 istanze


### 2b. Distribuzione della classe

Esaminiamo la distribuzione della variabile target `y`. Questo passaggio è cruciale perché
rivela il problema centrale dell'intero progetto: il **forte sbilanciamento di classe**. I
clienti che sottoscrivono il deposito (`yes`) sono una netta minoranza rispetto a chi rifiuta
(`no`). Conoscere questa proporzione è indispensabile per scegliere, nei task successivi, le
metriche di valutazione corrette (l'accuratezza da sola sarebbe fuorviante).

In [8]:
df["y"].value_counts()

y
no     36537
yes     4639
Name: count, dtype: int64

In [9]:
perc = round(df["y"].value_counts(normalize=True)["yes"] * 100, 2)
print(f"Proporzione 'yes': {perc}%")

Proporzione 'yes': 11.27%


### 2c. Valori `unknown`

Nel dataset i valori mancanti non sono codificati come `NaN`, ma con la stringa esplicita
**`unknown`** in diversi attributi nominali (Lezione 3). Prima di procedere è importante
**censirli** per quantificarne l'entità: alcune colonne ne contengono parecchi. In questa fase
li lasciamo **come sono** — `unknown` è di fatto una categoria informativa a sé (il fatto che
un'informazione manchi può essere predittivo) — ma è bene sapere dove si concentrano.

La tecnica di conteggio è **vettoriale**: il confronto `(df[nominali] == "unknown")` produce una
maschera booleana e `.sum()` conta i `True` colonna per colonna, senza alcun ciclo `for`.

In [10]:
# Censimento dei valori 'unknown' nelle colonne nominali (Lezione 3).
# Versione vettorizzata: il confronto (df == "unknown") produce una maschera
# booleana e .sum() conta i True colonna per colonna, senza cicli espliciti.
nominali = df.select_dtypes(include=["object", "string"]).columns

unknown_count = (df[nominali] == "unknown").sum()      # conteggio per colonna
unknown_count = unknown_count[unknown_count > 0]        # solo colonne con unknown

riepilogo_unknown = pd.DataFrame({
    "unknown": unknown_count,
    "%": (unknown_count / len(df) * 100).round(1),
})
print(riepilogo_unknown)

           unknown     %
job            330   0.8
marital         80   0.2
education     1730   4.2
default       8596  20.9
housing        990   2.4
loan           990   2.4


## 3. Codifica della classe

Trasformiamo la variabile target da stringa a intero: **`no` → 0** e **`yes` → 1**. La classe
positiva di interesse (la sottoscrizione, l'evento "raro" che vogliamo prevedere) è quindi
codificata come **`1`**. Questa codifica binaria è il formato atteso dai classificatori e
rende immediato il calcolo di metriche come precision e recall sulla classe positiva.

In [11]:
df["y"] = df["y"].map({"no": 0, "yes": 1})
print("Classe y codificata: no->0, yes->1 (yes = classe positiva)")

Classe y codificata: no->0, yes->1 (yes = classe positiva)


In [12]:
df["y"].value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

## 4. Estrazione dei due file
### 4a. `training.csv`

In [13]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [14]:
training = df.copy()
training.to_csv("../data/processed/training.csv", index=False)
print(f"Salvato training.csv: {training.shape[0]} istanze x {training.shape[1]} attributi")

Salvato training.csv: 41176 istanze x 21 attributi


### 4b. `manuale.csv`

Estraiamo ora il file `manuale.csv` richiesto dal **punto 1 della traccia** (10–15 campioni).
Costruiamo un insieme di **12 istanze deliberatamente bilanciate**: 6 della classe `yes` e 6
della classe `no`, campionate casualmente (con il seme fissato) e poi mescolate così che le
classi non risultino raggruppate.

**Due scelte di progetto importanti:**

- **Bilanciamento.** A differenza del dataset reale (fortemente sbilanciato), `manuale.csv` è
  bilanciato di proposito: serve a costruire e illustrare i classificatori a mano (Task 2) in
  un contesto pulito, dove entrambe le classi sono ugualmente rappresentate.
- **Sottoinsieme di feature.** Selezioniamo solo **9 attributi** (2 numerici e 7 nominali) tra
  i 20 disponibili. Sono attributi comprensibili e gestibili nel calcolo manuale: l'obiettivo
  del Task 2 non è la performance, ma mostrare *passo per passo* come funzionano gli algoritmi.

Si noti che `manuale.csv` viene salvato con il separatore di default (la **virgola**), mentre
`training.csv` usa il punto e virgola: una differenza da ricordare quando si ricaricano i file
nei task successivi.

In [15]:
feature_manuale = [
    "age", "campaign",                # numerici
    "job", "marital", "education",    # nominali
    "housing", "loan", "contact", "poutcome",
    "y",                              # classe
]

In [16]:
# 6 istanze 'yes' e 6 'no' -> set bilanciato (12 istanze)
yes_rows = df[df["y"] == 1].sample(n=6, random_state=SEED)
no_rows  = df[df["y"] == 0].sample(n=6, random_state=SEED)

In [17]:
manuale = pd.concat([yes_rows, no_rows])[feature_manuale]
# Mescoliamo l'ordine cosi' le classi non sono raggruppate
manuale = manuale.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [18]:
manuale.to_csv("../data/processed/manuale.csv", index=False)
print(f"Salvato manuale.csv: {manuale.shape[0]} istanze x {manuale.shape[1]} attributi")

Salvato manuale.csv: 12 istanze x 10 attributi


In [19]:
manuale

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## Riepilogo

Abbiamo prodotto:
- **`training.csv`** — dataset pulito completo, pronto per EDA e addestramento
- **`manuale.csv`** — 12 istanze bilanciate con attributi adatti ai calcoli a mano

**Prossimo passo (Task 2):** definire a mano i due classificatori (Naïve Bayes e
KNN) su `manuale.csv`, illustrare i passi per adattarli ai dati, implementarli in
Python e valutarne le prestazioni.
